## Блок тестовое задание

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [5]:
file_path = 'Данные для тестового задания.xlsx'
aud  = pd.read_excel(file_path, sheet_name='Данные об аудитории')
ab   = pd.read_excel(file_path, sheet_name='Данные АБ тестов')
lst  = pd.read_excel(file_path, sheet_name='Листеры')
aud.shape, ab.shape, lst.shape

((16814, 3), (2835, 4), (158, 6))

### 1 Чему равен MAU

In [9]:
november = aud[
    (aud['date'].dt.year == 2023) &
    (aud['date'].dt.month == 11)
]

mau = november['user_id'].nunique()

print('MAU за ноябрь 2023:', mau)

MAU за ноябрь 2023: 7639


### 2 Чему равен DAU

In [10]:
dau = aud.groupby('date')['user_id'].nunique().mean()
print('Средний DAU:', round(dau))

Средний DAU: 560


### 3 retention первого дня

In [11]:
day1 = set(aud[aud['date'] == '2023-11-01']['user_id'])
day2 = set(aud[aud['date'] == '2023-11-02']['user_id'])

retention = len(day1 & day2) / len(day1) * 100

print('Retention Day 1:', round(retention, 1), '%')

Retention Day 1: 26.6 %


### 5 Пользовательская конверсия

In [12]:
conversion = (
    aud[aud['view_adverts'] > 0]['user_id'].nunique()
    / aud['user_id'].nunique()
    * 100
)

print('Конверсия:', round(conversion, 1), '%')

Конверсия: 46.3 %


### 6 среднее количество просмотренных объявлений

In [13]:
avg_views = aud['view_adverts'].sum() / aud['user_id'].nunique()

print('Среднее количество просмотров:', round(avg_views, 1))

Среднее количество просмотров: 2.9


### 7 Расчёт NPS

In [14]:
promoters = 1200
detractors = 500
total = 2000

nps = (promoters / total - detractors / total) * 100

print('NPS:', round(nps), '%')

NPS: 35 %


### 8 Анализ трёх A/B-тестов

In [16]:
arpu = ab.groupby(
    ['experiment_num', 'experiment_group']
)['revenue'].agg(['count', 'sum', 'mean'])

arpu

count     sum        mean
experiment_num experiment_group                           
1              control             465  335944  722.460215
               test                480  319555  665.739583
2              control             465  327664  704.653763
               test                480  159806  332.929167
3              control             465  308391  663.206452
               test                480  479361  998.668750

In [17]:
for i in [1, 2, 3]:
    control = ab[
        (ab['experiment_num'] == i) &
        (ab['experiment_group'] == 'control')
    ]['revenue']

    test = ab[
        (ab['experiment_num'] == i) &
        (ab['experiment_group'] == 'test')
    ]['revenue']

    t_stat, p_value = stats.ttest_ind(
        control, test, equal_var=False
    )

    print(f'Эксперимент {i}: p-value = {p_value:.4f}')

Эксперимент 1: p-value = 0.6890
Эксперимент 2: p-value = 0.0011
Эксперимент 3: p-value = 0.0603


### 9 средний доход пользователя 

In [21]:
avg_revenue = lst.groupby('user_id')['revenue'].sum().mean()

print('Средний доход на пользователя:', round(avg_revenue, 2))

Средний доход на пользователя: 156.48


### 10 Медиана возраста пользователей

In [22]:
median_age = lst.drop_duplicates('user_id')['age'].median()

print('Медиана возраста:', median_age)

Медиана возраста: 28.0


### 18 вывод по результатам A/B-теста

In [23]:
from statsmodels.stats.proportion import proportions_ztest

payments = [1003, 1099]
visitors = [100047501, 100001055]

z_stat, p_value = proportions_ztest(payments, visitors)

print('Конверсия A:', round(payments[0] / visitors[0] * 100, 6), '%')
print('Конверсия B:', round(payments[1] / visitors[1] * 100, 6), '%')
print('p-value:', round(p_value, 4))

Конверсия A: 0.001003 %
Конверсия B: 0.001099 %
p-value: 0.0353
